# Second review coalition training noise

This notebook runs the first new experiment for Reviewer 1 comment 3. It reuses frozen production teacher banks and performs five paired empty/full coalition repetitions on outer seeds 11, 23, and 37 for each dataset. It never computes test predictions. Empty means supervised-only; full means uniform KD from all five frozen teachers.

Use a GPU runtime. Results are saved incrementally under `MyDrive/makd-har/results/revision-2/coalition-noise`. Run cells in order. Existing production folders are read only. A failed hash, split, or teacher validation-parity check stops the experiment. Smoke outputs are separate and cannot be reported as research results.

The final two cells run 30 student fits per dataset. Teacher loading and predictions happen once per bank and can be cached. Runtime hours are not yet measured. These repeated empty/full games are a noise diagnostic, not confidence intervals for every Shapley contribution.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT = Path('/content/drive/MyDrive/makd-har')
EXPERIMENT = PROJECT / 'revision_2'
assert (EXPERIMENT / 'frozen-inputs.json').is_file(), 'Wait for Google Drive desktop sync or check the project path.'
print('Project:', PROJECT)


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                       str(EXPERIMENT / 'pipeline_snapshot/requirements-colab.txt')])
# If Colab asks for a runtime restart, restart and rerun from the mount cell.
import tensorflow as tf
print('TensorFlow:', tf.__version__, 'Keras:', tf.keras.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('Select a GPU runtime before the training cells.')


In [ ]:
# Verify frozen code, datasets, source records and checkpoints. No training.
subprocess.run([sys.executable, str(EXPERIMENT / 'run_coalition_noise.py'),
                '--drive-root', str(PROJECT), '--dataset', 'all', '--verify-files-only'], check=True)


In [ ]:
# One-epoch smoke run per dataset. These outputs are NOT production evidence.
subprocess.run([sys.executable, str(EXPERIMENT / 'run_coalition_noise.py'),
                '--drive-root', str(PROJECT), '--dataset', 'all', '--smoke'], check=True)


In [ ]:
# UCI-HAR: three frozen teacher banks, five paired repetitions, 20-epoch limit.
subprocess.run([sys.executable, str(EXPERIMENT / 'run_coalition_noise.py'),
                '--drive-root', str(PROJECT), '--dataset', 'uci'], check=True)


In [ ]:
# Corrected WISDM startup. Use this cell for retries; completed UCI-HAR is preserved.
import subprocess, sys, json
from pathlib import Path
from datetime import datetime, timezone
PROJECT = Path('/content/drive/MyDrive/makd-har')
EXPERIMENT = PROJECT / 'revision_2'
# Preserve default-precision preflight caches if there are no research fits.
wisdm_root = PROJECT / 'results/revision-2/coalition-noise/wisdm'
startup_record = wisdm_root / 'deterministic-startup.json'
if wisdm_root.exists():
    previous_settings = json.loads(startup_record.read_text()) if startup_record.exists() else {}
    if previous_settings.get('tensor_float_32_execution') is not False:
        if list(wisdm_root.rglob('repetition-*.json')) or (wisdm_root / 'completed.json').exists():
            raise RuntimeError('Research fits already exist; do not mix precision modes.')
        archive = PROJECT / 'results/revision-2/diagnostics' / ('wisdm-preflight-default-precision-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S'))
        archive.parent.mkdir(parents=True, exist_ok=True)
        assert (PROJECT / 'results/revision-2').resolve() in wisdm_root.resolve().parents
        assert (PROJECT / 'results/revision-2').resolve() in archive.resolve().parents
        wisdm_root.rename(archive)
        print('Preserved unsuccessful preflight at:', archive, flush=True)
launcher_source = '"""Initialize deterministic operations before frozen-teacher validation."""\nimport argparse\nimport hashlib\nimport json\nimport runpy\nimport sys\nfrom pathlib import Path\n\n\ndef file_hash(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--drive-root\', type=Path, required=True)\n    parser.add_argument(\'--dataset\', choices=(\'wisdm\',), default=\'wisdm\')\n    parser.add_argument(\'--preflight-only\', action=\'store_true\')\n    args = parser.parse_args()\n    here = Path(__file__).resolve().parent\n    original = here / \'run_coalition_noise.py\'\n    frozen = json.loads((here / \'frozen-inputs.json\').read_text())\n    if file_hash(original) != frozen[\'runtime_files\'][\'run_coalition_noise.py\']:\n        raise RuntimeError(\'The original frozen runner has changed.\')\n    import tensorflow as tf\n    tf.keras.utils.set_random_seed(2026)\n    tf.config.experimental.enable_op_determinism()\n    tf.config.experimental.enable_tensor_float_32_execution(False)\n    startup = dict(\n        correction=\'deterministic startup before frozen-teacher inference\',\n        initial_seed=2026, deterministic_operations=True,\n        tensor_float_32_execution=False,\n        tensorflow=tf.__version__, keras=tf.keras.__version__,\n        original_runner_sha256=file_hash(original),\n        startup_launcher_sha256=file_hash(__file__),\n        frozen_inputs_sha256=file_hash(here / \'frozen-inputs.json\'),\n        note=\'Student training still resets each prescribed repetition seed. No tolerance, source weights, data, or epoch budget is changed.\',\n    )\n    record = args.drive_root / \'results/revision-2/coalition-noise/wisdm/deterministic-startup.json\'\n    record.parent.mkdir(parents=True, exist_ok=True)\n    if record.exists() and json.loads(record.read_text()) != startup:\n        raise RuntimeError(\'A different startup record already exists; do not mix configurations.\')\n    record.write_text(json.dumps(startup, indent=2, sort_keys=True), encoding=\'utf-8\')\n    print(\'Deterministic operations and full float32 precision enabled before teacher validation.\', flush=True)\n    sys.argv = [str(original), \'--drive-root\', str(args.drive_root), \'--dataset\', \'wisdm\']\n    if args.preflight_only:\n        sys.argv.append(\'--preflight-only\')\n    runpy.run_path(str(original), run_name=\'__main__\')\n\n\nif __name__ == \'__main__\':\n    main()\n'
launcher = EXPERIMENT / 'run_wisdm_noise_deterministic.py'
if launcher.exists() and launcher.read_text() != launcher_source:
    previous = launcher.with_name('run_wisdm_noise_deterministic-before-colab.py')
    if not previous.exists():
        previous.write_text(launcher.read_text())
launcher.write_text(launcher_source)
log_dir = PROJECT / 'results/revision-2/diagnostics'
log_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
for stage, extra in [('preflight', ['--preflight-only']), ('research', [])]:
    print('WISDM stage:', stage, flush=True)
    command = [sys.executable, '-u', str(launcher), '--drive-root', str(PROJECT), '--dataset', 'wisdm'] + extra
    with (log_dir / f'wisdm-full-float32-{stamp}-{stage}.log').open('w') as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)
print('Corrected WISDM run finished. Existing UCI-HAR results were not rerun.')


In [ ]:
import json
for dataset in ('uci', 'wisdm'):
    completed = PROJECT / 'results/revision-2/coalition-noise' / dataset / 'completed.json'
    assert completed.is_file(), f'{dataset} is incomplete; rerun its training cell to resume.'
    report = json.loads(completed.read_text())
    assert report['status'] == 'complete' and report['fits'] == 30 and report['test_evaluated'] is False
    print(dataset, 'COMPLETE:', report['fits'], 'student fits. Ready for OneDrive analysis.')


## After completion

Keep the output JSON files in Google Drive. The OneDrive analysis reads these results and compares within-bank utility variation with the raw Shapley scale already extracted from production. A result with large noise is still a valid finding. Do not change repetitions, seeds or protocols based on the observed result.

If a data or teacher-parity check fails, preserve the error and stop. Do not retrain the teachers or change the parser to make the check pass. A different teacher bank defines a different diagnostic and needs its own protocol.
